# 01 · Image analysis (MEM2, with MEM3 histograms and MEM4 spectra)

Explores the **development set only** (321 images, `data/split.json`) before any design.
All loading goes through `src.data`. Figures are also saved to `experiments/figures/`.

Spectrum and histogram cells use NumPy directly for now; they switch to `src.frequency.centered_spectrum` (MEM4) and `src.enhancement.histogram` (MEM3) once those are merged.

In [1]:
import collections, csv, pathlib, sys
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src import data

DATA = ROOT / 'data/raw/concrete_crack'
FIG = ROOT / 'experiments/figures'
RES = ROOT / 'experiments/results'
L = 1008  # working size (long side), from Section 4
PAIRS = {img.stem: (img, gt) for img, gt in data.list_pairs(DATA)}
DEV = data.load_split(ROOT / 'data/split.json')['dev']
print(len(DEV), 'dev images')

321 dev images


## 1 · Size, data type, range, crack fraction

In [2]:
sizes, crack_pct, gray_mean, gray_std = collections.Counter(), [], [], []
for n in DEV:
    img = cv2.imread(str(PAIRS[n][0]), cv2.IMREAD_COLOR)
    sizes[img.shape] += 1
    gray = data.load_image(PAIRS[n][0], size=L)
    gt = data.load_ground_truth(PAIRS[n][1], size=L)
    crack_pct.append(100 * gt.mean()); gray_mean.append(gray.mean()); gray_std.append(gray.std())
crack_pct, gray_mean, gray_std = map(np.array, (crack_pct, gray_mean, gray_std))
print('raw shapes (H, W, C):', dict(sizes), '| dtype uint8, range 0-255')
print(f'crack % at L={L}: mean {crack_pct.mean():.2f}, median {np.median(crack_pct):.2f},' f' min {crack_pct.min():.2f}, max {crack_pct.max():.2f}')
print(f'gray mean {gray_mean.mean():.1f} (min {gray_mean.min():.0f}, max {gray_mean.max():.0f}); global contrast (std) median {np.median(gray_std):.1f}')

raw shapes (H, W, C): {(3264, 2448, 3): 6, (3024, 4032, 3): 173, (2448, 3264, 3): 8, (4032, 3024, 3): 134} | dtype uint8, range 0-255
crack % at L=1008: mean 1.36, median 1.17, min 0.41, max 8.70
gray mean 179.3 (min 124, max 200); global contrast (std) median 19.8


## 2 · Typical vs difficult images and their histograms
Typical = dev image with median contrast. Difficult examples (dev only): 148 hard shadow, 385 dark corners, 109 low contrast.

In [3]:
typical = DEV[int(np.argsort(gray_std)[len(DEV) // 2])]
SHOW = {typical: 'typical', '148': 'shadow', '385': 'dark corners', '109': 'low contrast'}
fig, ax = plt.subplots(3, 4, figsize=(16, 10))
for j, (n, label) in enumerate(SHOW.items()):
    rgb = data.load_image(PAIRS[n][0], mode='rgb', size=L)
    gray = data.load_image(PAIRS[n][0], size=L)
    gt = data.load_ground_truth(PAIRS[n][1], size=L)
    over = rgb.copy(); over[gt] = (255, 0, 0)
    ax[0, j].imshow(rgb); ax[0, j].set_title(f'{n}: {label}')
    ax[1, j].imshow(over); ax[1, j].set_title('ground truth (red)')
    ax[2, j].hist(gray[~gt].ravel(), bins=256, range=(0, 256), density=True, alpha=.6, label='background')
    ax[2, j].hist(gray[gt].ravel(), bins=256, range=(0, 256), density=True, alpha=.6, label='crack')
    ax[2, j].axvline(127, color='k', ls='--', lw=1); ax[2, j].legend()
    for a in ax[:2, j]: a.axis('off')
fig.tight_layout(); fig.savefig(FIG / 'analysis_typical_vs_difficult.png', dpi=80)

## 3 · Crack vs background intensity over the whole dev set
How separable are cracks by gray level alone? (Pooled sample, 20k pixels per class per image.)

In [4]:
rng = np.random.default_rng(0)
crack_px, bg_px = [], []
for n in DEV:
    gray = data.load_image(PAIRS[n][0], size=L)
    gt = data.load_ground_truth(PAIRS[n][1], size=L)
    crack_px.append(rng.choice(gray[gt], min(20000, gt.sum())))
    bg_px.append(rng.choice(gray[~gt], 20000))
crack_px, bg_px = np.concatenate(crack_px), np.concatenate(bg_px)
print(f'crack gray: median {np.median(crack_px):.0f}, 90th pct {np.percentile(crack_px, 90):.0f}')
print(f'background gray: median {np.median(bg_px):.0f}, 10th pct {np.percentile(bg_px, 10):.0f}')
for t in (100, 127, 150):
    print(f'T={t}: crack pixels below T {100 * (crack_px < t).mean():.1f}%, '
          f'background pixels below T {100 * (bg_px < t).mean():.2f}%')

crack gray: median 111, 90th pct 177
background gray: median 183, 10th pct 147
T=100: crack pixels below T 42.2%, background pixels below T 0.41%
T=127: crack pixels below T 59.8%, background pixels below T 3.55%
T=150: crack pixels below T 73.9%, background pixels below T 11.05%


## 4 · Uneven illumination
Background estimate = Gaussian blur with sigma = 5% of the long side. Unevenness = std / mean of that estimate.

In [5]:
def background(gray, frac=0.05):
    return cv2.GaussianBlur(gray.astype(np.float64), (0, 0), frac * max(gray.shape))
unevenness = np.array([(lambda b: b.std() / b.mean())(background(data.load_image(PAIRS[n][0], size=L))) for n in DEV])
print(f'illumination unevenness: median {np.median(unevenness):.3f}, 90th pct {np.percentile(unevenness, 90):.3f}, max {unevenness.max():.3f}')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, n in zip(ax[:2], (typical, '148')):
    a.imshow(background(data.load_image(PAIRS[n][0], size=L)), cmap='gray', vmin=0, vmax=255)
    a.set_title(f'{n}: estimated background'); a.axis('off')
    ax[2].plot(background(data.load_image(PAIRS[n][0], size=L)).mean(axis=0), label=n)
ax[2].set_title('column profile of the background'); ax[2].legend()
fig.tight_layout(); fig.savefig(FIG / 'analysis_illumination.png', dpi=80)

illumination unevenness: median 0.081, 90th pct 0.134, max 0.231


## 5 · Noise
Noise estimate = std of (image − 3×3 median) in pixels away from cracks.

In [6]:
noise = []
for n in DEV:
    gray = data.load_image(PAIRS[n][0], size=L)
    gt = cv2.dilate(data.load_ground_truth(PAIRS[n][1], size=L).astype(np.uint8), np.ones((9, 9), np.uint8)) > 0
    resid = gray.astype(np.float64) - cv2.medianBlur(gray, 3)
    noise.append(resid[~gt].std())
noise = np.array(noise)
print(f'noise std (gray levels): median {np.median(noise):.2f}, max {noise.max():.2f}')

noise std (gray levels): median 3.42, max 8.68


## 6 · Centred log-magnitude spectrum

In [7]:
def centred_log_spectrum(gray):
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray.astype(np.float64)))))
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, n in zip(ax, (typical, '148', '109')):
    a.imshow(centred_log_spectrum(data.load_image(PAIRS[n][0], size=L)), cmap='magma')
    a.set_title(f'{n}: log(1+|F|), centred'); a.axis('off')
fig.tight_layout(); fig.savefig(FIG / 'analysis_spectrum.png', dpi=80)

## 7 · Resolution and gray-level study (all 321 dev images)
Produced by `python -m src.pipeline --method baseline_trivial --size S` and `--size 1008 --levels K`; per-image CSVs in `experiments/results/`.

In [8]:
def mean_std(path):
    v = np.array([float(r['iou']) for r in csv.DictReader(open(path, encoding='utf-8'))])
    return v.mean(), v.std()
print('long side  IoU mean +/- std')
for s in (4032, 2016, 1008, 504):
    m, sd = mean_std(RES / f'base_res_size{s}.csv'); print(f'{s:9d}  {m:.4f} +/- {sd:.4f}')
print('Trivial baseline: every power-of-2 level count keeps a bin edge at 128, so')
print('threshold 127 gives the same IoU at all levels (base_gray_levels*.csv).')
print('Final pipeline, gray levels (L=1008, study_gray_levels.csv):')
for r in csv.DictReader(open(RES / 'study_gray_levels.csv', encoding='utf-8')):
    print(f"{r['variant']:>11}  {float(r['iou_mean']):.4f} +/- {float(r['iou_std']):.4f}")

long side  IoU mean +/- std
     4032  0.4342 +/- 0.1929
     2016  0.4310 +/- 0.1901
     1008  0.4416 +/- 0.1932
      504  0.4544 +/- 0.1998
Trivial baseline: every power-of-2 level count keeps a bin edge at 128, so
threshold 127 gives the same IoU at all levels (base_gray_levels*.csv).
Final pipeline, gray levels (L=1008, study_gray_levels.csv):
 256 levels  0.6657 +/- 0.1201
  64 levels  0.6669 +/- 0.1195
  32 levels  0.6666 +/- 0.1196
  16 levels  0.6652 +/- 0.1206
   8 levels  0.6001 +/- 0.1246
   4 levels  0.2188 +/- 0.1023


In [9]:
n = '148'
full = data.load_image(PAIRS[n][0])
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, s in zip(ax[0], (4032, 2016, 1008, 504)):
    img = data.load_image(PAIRS[n][0], size=s)
    crop = img[: img.shape[0] // 4, : img.shape[1] // 4]
    a.imshow(crop, cmap='gray', vmin=0, vmax=255); a.set_title(f'long side {s} (crop)')
for a, k in zip(ax[1], (256, 16, 4, 2)):
    a.imshow(data.quantize_gray_levels(data.load_image(PAIRS[n][0], size=L), k), cmap='gray', vmin=0, vmax=255)
    a.set_title(f'{k} gray levels')
for a in ax.ravel(): a.axis('off')
fig.tight_layout(); fig.savefig(FIG / 'analysis_resolution_gray_levels.png', dpi=80)

## 8 · Findings → design decisions

| # | Finding (dev set only) | Design decision |
|---|---|---|
| 1 | Cracks are **darker** than the background; crack and background gray levels overlap (Section 3) | Every threshold marks pixels **below** T; a single global T cannot separate them on all images |
| 2 | Cracks cover only ~1.4 % of pixels; Otsu on the raw image splits background texture (simple baseline IoU 0.07) | Primary metric IoU (not accuracy); avoid global histogram thresholds on raw images |
| 3 | Shadows and dark corners change the background by tens of gray levels (Section 4) | Local (adaptive) threshold; frequency-domain background flattening tested (gain small once the threshold is local, ablation AB6) |
| 4 | Fine surface texture and pores give many small dark spots | Gaussian smoothing before thresholding; object selection by area and circularity after it |
| 5 | Little sensor noise (residual std median 3.4 gray levels, max 8.7) | Noise robustness tested with added Gaussian noise; median filtering not needed for the raw data |
| 6 | Spectrum energy concentrated at low frequencies; crack lines add a streak perpendicular to the crack | Low-pass cutoff for flattening chosen small (D0 = 12 on the 2× padded grid) |
| 7 | Resolution: IoU 0.454 at 504 px, 0.442 at 1008, 0.431 at 2016, 0.434 at 4032 (trivial baseline) | Working size 1008: within noise of the best, keeps 4× the pixels of 504 for thin cracks, fast enough |
| 8 | Gray levels (final pipeline): 16 levels lose nothing, 8 → 0.600, 4 → 0.219 | 16 gray levels is the lowest workable setting; we keep 256 |
